# LLM CoT Evaluation Pipeline
## Vietnamese Math Q&A with Llama-3.2-1B

This notebook implements a complete evaluation pipeline for comparing Direct (non-CoT) vs Chain-of-Thought (CoT) responses from Llama model on Vietnamese math problems.

## 1. Setup & Imports

In [ ]:
import gc
import os
import re
from pathlib import Path
from textwrap import dedent

import torch
from datasets import Dataset
from dotenv import load_dotenv
from huggingface_hub import login
from openpyxl import load_workbook
from transformers import logging as hf_logging, pipeline

# Suppress HuggingFace warnings
hf_logging.set_verbosity_error()

# Load environment variables
load_dotenv()

## 2. Configuration & Constants

In [ ]:
# --- Dataset & Model Configuration ---
DATASET_FILE_PATH = "data/sample_dataset.xlsx"
DATASET_NAME = DATASET_FILE_PATH  # Backward-compatible alias
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
BATCH_SIZE = 5
PIPE_BATCH_SIZE = 5

# --- Inference Parameters ---
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.1  # Lower for math accuracy
TOP_P = 0.95

# --- Prompt Markers ---
SOLUTION_START = "####"
SOLUTION_END = ""
REASONING_START = "<thought>"
REASONING_END = "</thought>"

print(f"Configuration:")
print(f"  Dataset: {DATASET_FILE_PATH}")
print(f"  Model: {MODEL_ID}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Device: {'GPU (bfloat16)' if torch.cuda.is_available() else 'CPU (float32)'}")

## 3. Prompt Definition

In [ ]:
# TODO (1): Viết system prompt cho chế độ trả lời TRỰC TIẾP (không có Chain-of-Thought).
#
#   Prompt cần:
#     1. Giao vai trò rõ ràng cho model (agent description).
#     2. Mô tả đầu vào – bài toán bằng tiếng Việt.
#     3. Chỉ định định dạng đầu ra – chỉ xuất đáp án số, đặt sau SOLUTION_START.
#     4. Cấm model giải thích hay trình bày bước làm.
#
#   Lưu ý: dùng f-string để nhúng SOLUTION_START / SOLUTION_END vào prompt.
DIRECT_PROMPT = f"""
"""

# TODO (2): Viết system prompt cho chế độ Chain-of-Thought (CoT).
#
#   Prompt cần:
#     1. Giao vai trò như TODO (1).
#     2. Yêu cầu model suy luận từng bước bằng tiếng Việt TRƯỚC khi đưa ra đáp án.
#     3. Phần suy luận phải được bọc giữa REASONING_START và REASONING_END.
#     4. Đáp án số cuối cùng đặt sau SOLUTION_START (và trước SOLUTION_END nếu có).
#
#   Gợi ý cấu trúc:
#     [Vai trò]
#     [Mô tả đầu vào]
#     Bước 1 – Suy luận: đặt trong {REASONING_START} ... {REASONING_END}
#     Bước 2 – Đáp án:   đặt sau {SOLUTION_START}
COT_PROMPT = f"""
"""

print("Prompts defined:")
print(f"  - DIRECT_PROMPT: {len(DIRECT_PROMPT)} characters")
print(f"  - COT_PROMPT: {len(COT_PROMPT)} characters")

## 4. Utility Functions

In [ ]:
def extract_answer(text: str) -> str:
    """Extract the answer number from the response text.
    
    Prefer the final `####{answer}` pattern, then fallback to other numeric forms.
    """
    # Pattern: #### {number} at end of text (highest priority)
    m = re.search(r"####\s*([+-]?[\d,]+(?:\.\d+)?\s*)$", text)
    if m:
        return m.group(1).strip().replace(",", "")

    # Pattern: #### {number} anywhere in text
    m = re.search(r"####\s*([+-]?[\d,]+(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    # Pattern: "Đáp án là: {number}"
    m = re.search(r"Đáp án là:\s*([+-]?[\d,]+(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    # Pattern: "The answer is {number}"
    m = re.search(r"[Tt]he answer is\s*([+-]?[\d,]+(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    # Fallback: extract last number in text
    nums = re.findall(r"[+-]?[\d,]+(?:\.\d+)?", text)
    return nums[-1].replace(",", "") if nums else ""


def decode_pipe_output(outputs):
    """Decode outputs from HuggingFace pipeline to list of text strings."""
    texts = []
    for out in outputs:
        if isinstance(out, dict) and "generated_text" in out:
            texts.append(out["generated_text"])
        elif isinstance(out, list) and len(out) > 0 and "generated_text" in out[0]:
            texts.append(out[0]["generated_text"])
        elif isinstance(out, str):
            texts.append(out)
        else:
            texts.append(str(out))
    return texts

## 5. Dataset Loading Functions

In [ ]:
def _resolve_dataset_path(dataset_path: str) -> Path:
    """Resolve dataset path relative to notebook location."""
    path = Path(dataset_path)
    if path.is_absolute():
        return path
    # In notebook context, use current working directory
    return Path.cwd() / path


def load_dataset_from_excel(dataset_path: str):
    """Load the local Excel dataset and normalize its columns."""
    path = _resolve_dataset_path(dataset_path)
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.active

    headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
    if "query_vi" not in headers:
        raise ValueError(f"Missing required column 'query_vi' in {path}")

    response_header = next(
        (h for h in headers if isinstance(h, str) and h.startswith("response_vi")),
        None,
    )
    if response_header is None:
        raise ValueError(f"Missing required response column in {path}")

    rows = []
    for values in ws.iter_rows(min_row=2, values_only=True):
        row = {}
        for header, value in zip(headers, values):
            if header == "query_vi":
                row["query_vi"] = value
            elif header == response_header:
                row["response_vi"] = value
        if row:
            rows.append(row)

    wb.close()
    return Dataset.from_list(rows)


def add_ground_truth(ds):
    """Extract ground truth answers from the response column."""
    return ds.map(
        lambda example: {"ground_truth": extract_answer(example["response_vi"])},
        remove_columns=[],
    )

## 6. Pipeline Building & Text Generation

In [ ]:
def get_hf_token():
    """Get HuggingFace token from environment."""
    return os.getenv("HF_TOKEN")


def build_pipeline():
    """Build and return HuggingFace pipeline with proper device and auth."""
    hf_token = get_hf_token()
    if hf_token:
        login(token=hf_token)
    else:
        print("Note: HF_TOKEN not found in environment. Please login manually.")

    device = 0 if torch.cuda.is_available() else -1
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    pipe = pipeline(
        "text-generation",
        model=MODEL_ID,
        device=device,
        dtype=dtype,
    )

    # Llama-style tokenizers often do not define a pad token by default.
    # Batched generation requires one, so reuse EOS and left-pad the inputs.
    if pipe.tokenizer.pad_token_id is None:
        pipe.tokenizer.pad_token = pipe.tokenizer.eos_token
        pipe.tokenizer.pad_token_id = pipe.tokenizer.eos_token_id
    pipe.tokenizer.padding_side = "left"

    return pipe


def generate_text_with_prompt(pipe, prompts, system_prompt: str, batch_size: int = PIPE_BATCH_SIZE):
    """Generate text using chat template with system prompt.

    Args:
        pipe: HuggingFace pipeline
        prompts: List of user prompts
        system_prompt: System message for the model
        batch_size: Batch size for pipeline

    Returns:
        List of generated text strings
    """
    formatted_prompts = []
    for user_prompt in prompts:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_str = pipe.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        formatted_prompts.append(prompt_str)

    outputs = pipe(
        formatted_prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        return_full_text=False,
        batch_size=batch_size,
    )

    return decode_pipe_output(outputs)

## 7. Main Evaluation Pipeline

In [ ]:
def run():
    print("=== Starting Evaluation Pipeline ===\n")

    # --- Load and prepare dataset ---
    print(f"1. Loading dataset from {DATASET_FILE_PATH}...")
    ds = load_dataset_from_excel(DATASET_FILE_PATH)
    print(f"   Dataset loaded with {len(ds)} samples")
    print("   Using the prepared 15-row dataset directly; no further sampling.\n")

    # Add ground truth extracted from response_vi
    ds = add_ground_truth(ds)

    # --- Build pipeline ---
    print("\n2. Building HuggingFace pipeline...")
    pipe = build_pipeline()
    print(f"   Model: {MODEL_ID}")
    print(f"   Device: {'GPU' if torch.cuda.is_available() else 'CPU'}\n")

    # --- Run evaluation ---
    print("3. Running batch evaluation...")
    print(f"   Total samples: {len(ds)}, Batch size: {BATCH_SIZE}\n")

    # Process batches
    for batch_idx, start in enumerate(range(0, len(ds), BATCH_SIZE), start=1):
        end = min(start + BATCH_SIZE, len(ds))
        batch = ds.select(range(start, end))

        queries = list(batch["query_vi"])
        ground_truths = list(batch["ground_truth"])

        # Generate responses: no-CoT and with-CoT
        print(f"   Batch {batch_idx}: Generating non-CoT responses...")
        no_cot_texts = generate_text_with_prompt(
            pipe, queries, DIRECT_PROMPT, batch_size=BATCH_SIZE
        )

        print(f"   Batch {batch_idx}: Generating CoT responses...")
        cot_texts = generate_text_with_prompt(
            pipe, queries, COT_PROMPT, batch_size=BATCH_SIZE
        )

        for idx, (q, gt, no_text, cot_text) in enumerate(
            zip(queries, ground_truths, no_cot_texts, cot_texts),
            start=start + 1,
        ):
            ans_no = extract_answer(no_text)
            ans_cot = extract_answer(cot_text)

            print(f"\n   Sample {idx}")
            print(f"     query_vi: {q}")
            print(f"     ground_truth: {gt}")
            print(f"     non-CoT answer: {ans_no}")
            print(f"     CoT answer: {ans_cot}")

        # Clean up memory
        del no_cot_texts, cot_texts
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --- Summary ---
    print("\n4. Final Summary")
    print(f"   Total samples processed: {len(ds)}")
    print("\n=== Evaluation Complete ===")

## 8. Execute Evaluation

In [ ]:
# Run the evaluation pipeline
run()